# Notebook 04 — Customer Risk Segmentation & Retention Priority Model
## Telecom Customer Churn and Retention Intelligence Dashboard

---

**Purpose:** Build an explainable, multi-factor churn risk scoring algorithm (0-100 score), assign customers to risk tiers (`High Risk`, `Medium Risk`, `Low Risk`), calculate customer value tiers, and create an actionable Retention Priority Matrix.

**Input:** `data/processed/telco_cleaned.csv`

**Output:** `data/processed/telco_segmented.csv`

**Key Scoring Rules (Total: 100 Points):**
- **Contract Type:** Month-to-month (+30 pts), One year (+10 pts), Two year (0 pts)
- **Tenure Lifecycle:** 0-12m (+25 pts), 13-24m (+15 pts), 25-48m (+5 pts), 49-72m (0 pts)
- **Internet Service:** Fiber optic (+15 pts), DSL (+5 pts), No Internet (0 pts)
- **Tech Support:** No tech support on internet (+10 pts)
- **Online Security:** No online security on internet (+10 pts)
- **Payment Method:** Electronic check (+10 pts)

---

## Section 1 — Import Libraries and Load Enriched Dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

PROJECT_ROOT = Path.cwd().parent
PROCESSED_FILE = PROJECT_ROOT / 'data' / 'processed' / 'telco_cleaned.csv'

df = pd.read_csv(PROCESSED_FILE)
print(f'Processed dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Section 2 — Multi-Factor Churn Risk Scoring Algorithm (0-100 Points)

**Why:** Rather than a black-box model, an explainable point-based risk scoring system allows business stakeholders, retention agents, and dashboard users to see exactly *why* a customer is at risk.

In [ ]:
def compute_risk_score(row):
    score = 0
    
    # 1. Contract Commitment (Max 30 pts)
    if row['contract'] == 'Month-to-month':
        score += 30
    elif row['contract'] == 'One year':
        score += 10
    
    # 2. Tenure Lifecycle (Max 25 pts)
    if row['tenure'] <= 12:
        score += 25
    elif row['tenure'] <= 24:
        score += 15
    elif row['tenure'] <= 48:
        score += 5
        
    # 3. Internet Service Type (Max 15 pts)
    if row['internet_service'] == 'Fiber optic':
        score += 15
    elif row['internet_service'] == 'DSL':
        score += 5
        
    # 4. Tech Support Protection (Max 10 pts)
    if row['internet_service'] != 'No' and row['tech_support'] == 'No':
        score += 10
        
    # 5. Online Security Protection (Max 10 pts)
    if row['internet_service'] != 'No' and row['online_security'] == 'No':
        score += 10
        
    # 6. Payment Method Friction (Max 10 pts)
    if row['payment_method'] == 'Electronic check':
        score += 10
        
    return score

df['risk_score'] = df.apply(compute_risk_score, axis=1)

print('Risk Score Summary:')
display(df['risk_score'].describe())

## Section 3 — Assign Risk Tiers (`risk_tier`)

**Tiers:**
- `High Risk` (Score >= 60)
- `Medium Risk` (Score 30 - 59)
- `Low Risk` (Score < 30)

In [ ]:
def assign_risk_tier(score):
    if score >= 60:
        return 'High Risk'
    elif score >= 30:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['risk_tier'] = df['risk_score'].apply(assign_risk_tier)

print('Risk Tier Performance vs Actual Churn Rate:')
tier_perf = df.groupby('risk_tier').agg(
    customer_count=('customer_id', 'count'),
    avg_score=('risk_score', 'mean'),
    actual_churners=('churn_flag', 'sum'),
    actual_churn_rate=('churn_flag', lambda x: round(x.mean() * 100, 2))
).reindex(['High Risk', 'Medium Risk', 'Low Risk']).reset_index()

display(tier_perf)

## Section 4 — Customer Value Tiers (`customer_value_tier`)

**Why:** Not all customers generate equal revenue. Retaining a $100/mo high-tenure customer is far more critical than a $20/mo low-tenure customer.

In [ ]:
def assign_value_tier(row):
    if row['monthly_charges'] >= 70 and row['tenure'] >= 24:
        return 'High Value'
    elif row['monthly_charges'] >= 50 or row['tenure'] >= 12:
        return 'Medium Value'
    else:
        return 'Low Value'

df['customer_value_tier'] = df.apply(assign_value_tier, axis=1)

print('Customer Value Tier Breakdown:')
value_summary = df.groupby('customer_value_tier').agg(
    customers=('customer_id', 'count'),
    avg_monthly_rev=('monthly_charges', 'mean'),
    avg_tenure=('tenure', 'mean'),
    churn_rate=('churn_flag', lambda x: round(x.mean() * 100, 2))
).reset_index()

display(value_summary)

## Section 5 — Retention Priority Matrix (`retention_priority`)

**Why:** Combining **Risk Tier** with **Customer Value Tier** creates a 2x2 action matrix for retention budgets.

In [ ]:
def assign_retention_priority(row):
    risk = row['risk_tier']
    val = row['customer_value_tier']
    
    if risk == 'High Risk' and val == 'High Value':
        return '1. Critical Save (VIP Risk)'
    elif risk == 'High Risk' and val in ['Medium Value', 'Low Value']:
        return '2. Urgent Retention'
    elif risk == 'Medium Risk' and val == 'High Value':
        return '3. Proactive Nurture'
    elif risk == 'Medium Risk':
        return '4. Standard Engagement'
    else:
        return '5. Low Touch / Stable'

df['retention_priority'] = df.apply(assign_retention_priority, axis=1)

print('Retention Priority Distribution & Financial Impact:')
priority_summary = df.groupby('retention_priority').agg(
    total_customers=('customer_id', 'count'),
    monthly_revenue_at_risk=('monthly_charges', 'sum'),
    avg_churn_risk_score=('risk_score', 'mean'),
    actual_churn_rate=('churn_flag', lambda x: round(x.mean() * 100, 2))
).reset_index()

display(priority_summary)

## Section 6 — Recommended Action Playbook Mapping

**Why:** Operationalize findings by assigning specific marketing/retention actions to each priority group.

In [ ]:
action_playbook = {
    '1. Critical Save (VIP Risk)': 'Direct Account Manager Call + 1-Yr Contract Discount + Free Tech Support Bundle',
    '2. Urgent Retention': 'Automated Onboarding Support + Autopay $5 Discount Incentive',
    '3. Proactive Nurture': 'Loyalty Reward + Complimentary Security Suite Upgrade',
    '4. Standard Engagement': 'Quarterly Satisfaction Survey + Self-Service Portal Promotion',
    '5. Low Touch / Stable': 'Standard Billing Notifications + Referral Program Invitations'
}

df['recommended_action'] = df['retention_priority'].map(action_playbook)

print('Sample Recommendations:')
display(df[['customer_id', 'risk_score', 'risk_tier', 'customer_value_tier', 'retention_priority', 'recommended_action']].head(8))

## Section 7 — Save Segmented Dataset

**Why:** Export the final segmented dataset to `data/processed/telco_segmented.csv`.

In [ ]:
output_segmented_file = PROJECT_ROOT / 'data' / 'processed' / 'telco_segmented.csv'
df.to_csv(output_segmented_file, index=False)

print(f'Segmented dataset saved successfully to:')
print(f'  {output_segmented_file}')
print(f'Final Segmented Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

---

## Next Step

Proceed to **`05_eda_churn_analysis.ipynb`** (Phase 7) for in-depth statistical exploratory data analysis, correlation heatmaps, churn driver deep dives, and visual asset generation.

---
**Customer Risk Segmentation Phase Complete.**